In [15]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

import numpy as np

In [21]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
import pandas as pd

In [2]:
path = "data/heart.csv"
df = pd.read_csv(path)
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [3]:
df.nunique()

Age                50
Sex                 2
ChestPainType       4
RestingBP          67
Cholesterol       222
FastingBS           2
RestingECG          3
MaxHR             119
ExerciseAngina      2
Oldpeak            53
ST_Slope            3
HeartDisease        2
dtype: int64

In [4]:
df.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='object')

In [6]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [8]:
dfle = df

dfle.Sex=le.fit_transform(dfle.Sex)
dfle.ChestPainType=le.fit_transform(dfle.ChestPainType)
dfle.RestingECG=le.fit_transform(dfle.RestingECG)
dfle.ExerciseAngina=le.fit_transform(dfle.ExerciseAngina)
dfle.ST_Slope=le.fit_transform(dfle.ST_Slope)

dfle

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,1,1,140,289,0,1,172,0,0.0,2,0
1,49,0,2,160,180,0,1,156,0,1.0,1,1
2,37,1,1,130,283,0,2,98,0,0.0,2,0
3,48,0,0,138,214,0,1,108,1,1.5,1,1
4,54,1,2,150,195,0,1,122,0,0.0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,3,110,264,0,1,132,0,1.2,1,1
914,68,1,0,144,193,1,1,141,0,3.4,1,1
915,57,1,0,130,131,0,1,115,1,1.2,1,1
916,57,0,1,130,236,0,0,174,0,0.0,1,1


In [11]:
dfle.ChestPainType.unique()

array([1, 2, 0, 3])

In [12]:
x = df.drop(columns='HeartDisease')
y = df.HeartDisease

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
x_scaled

array([[-1.4331398 ,  0.51595242,  0.22903206, ..., -0.8235563 ,
        -0.83243239,  1.05211381],
       [-0.47848359, -1.93816322,  1.27505906, ..., -0.8235563 ,
         0.10566353, -0.59607813],
       [-1.75135854,  0.51595242,  0.22903206, ..., -0.8235563 ,
        -0.83243239,  1.05211381],
       ...,
       [ 0.37009972,  0.51595242, -0.81699495, ...,  1.21424608,
         0.29328271, -0.59607813],
       [ 0.37009972, -1.93816322,  0.22903206, ..., -0.8235563 ,
        -0.83243239, -0.59607813],
       [-1.64528563,  0.51595242,  1.27505906, ..., -0.8235563 ,
        -0.83243239,  1.05211381]])

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=30)

In [19]:
model_params = {
    'svm': {
        'model': SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear',multi_class='auto'),
        'params': {
            'C': [1,5,10],
        }
    },
    'GaussianNB':{
        'model':GaussianNB(),
        'params':{

        }
    },
    'DecisionTree':{
        'model':DecisionTreeClassifier(),
        'params':{
            'criterion': ['gini','entropy'],
        }
    }

}

In [22]:
scores = []

for model_name,mp in model_params.items():

    clf = GridSearchCV(mp['model'],mp['params'], cv=5, return_train_score=False)
    clf.fit(X_train,y_train)

    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

scoredf = pd.DataFrame(scores,columns=['model','best_score','best_params'])
scoredf

,model,best_score,best_params
0,svm,0.867897,"{'C': 1, 'kernel': 'rbf'}"
1,random_forest,0.844721,{'n_estimators': 10}
2,logistic_regression,0.851533,{'C': 5}
3,GaussianNB,0.854235,{}
4,DecisionTree,0.790187,{'criterion': 'entropy'}


In [24]:
modelsvm = SVC(kernel='rbf',C=1)
modelsvm.fit(X_train,y_train)
modelsvm.score(X_test,y_test)

0.875

In [25]:
x.shape

(918, 11)

In [26]:
from sklearn.decomposition import PCA

pca = PCA(0.95)

x_pca = pca.fit_transform(x)
x_pca.shape

(918, 2)

In [27]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(x_pca, y, test_size=0.2, random_state=30)

In [28]:
modelsvm = SVC(kernel='rbf',C=1)
modelsvm.fit(X_train_pca,y_train)
modelsvm.score(X_test_pca,y_test)

0.7445652173913043

## Sources

Source: [Codebasics](https://github.com/codebasics/py).